# Install Dependencies

**CRITICAL: El orden de instalación importa en Blackwell (sm_120).**

1. PyTorch con cu128 **PRIMERO** (stable no soporta sm_120)
2. triton >= 3.3.1
3. bitsandbytes
4. unsloth (trae peft, transformers, accelerate)
5. trl, datasets, anthropic, pydantic

Si se instala en otro orden, vLLM o unsloth pueden reinstalar torch con cu126 y romper Blackwell.

**Nota Windows:** usar `dataset_num_proc=1` en todos los scripts de training.

## 1. Install PyTorch (cu128 — Blackwell)

In [1]:
# If stable PyTorch already supports cu128 by the time you run this,
# you can remove --pre and use the stable index instead.
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

Looking in indexes: https://download.pytorch.org/whl/nightly/cu128
   ---------------------------------------- 0.0/2.8 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.8 GB 16.9 MB/s eta 0:02:44
   ---------------------------------------- 0.0/2.8 GB 8.0 MB/s eta 0:05:45
   ---------------------------------------- 0.0/2.8 GB 8.7 MB/s eta 0:05:18
   ---------------------------------------- 0.0/2.8 GB 8.6 MB/s eta 0:05:22
   ---------------------------------------- 0.0/2.8 GB 10.1 MB/s eta 0:04:34
   ---------------------------------------- 0.0/2.8 GB 13.8 MB/s eta 0:03:19
   ---------------------------------------- 0.0/2.8 GB 12.9 MB/s eta 0:03:33
   ---------------------------------------- 0.0/2.8 GB 11.9 MB/s eta 0:03:52
   ---------------------------------------- 0.0/2.8 GB 11.2 MB/s eta 0:04:04
   ---------------------------------------- 0.0/2.8 GB 11.3 MB/s eta 0:04:03
   ---------------------------------------- 0.0/2.8 GB 11.8 MB/s eta 0:03:52
   -----------------

## 2. Verify PyTorch + Blackwell

In [2]:
import torch

assert torch.cuda.is_available(), "CUDA not available after install!"
cap = torch.cuda.get_device_capability(0)
print(f"PyTorch {torch.__version__}")
print(f"CUDA {torch.version.cuda}")
print(f"Compute capability: {cap[0]}.{cap[1]} (sm_{cap[0]}{cap[1]}0)")

if cap[0] < 12:
    print("\n⚠ sm_120 NOT detected. This torch build may not support Blackwell.")
    print("  Ensure you installed from the cu128 nightly index.")
else:
    print("\n✓ Blackwell OK")

PyTorch 2.12.0.dev20260323+cu128
CUDA 12.8
Compute capability: 12.0 (sm_1200)

✓ Blackwell OK


## 3. Install triton

In [3]:
# triton >= 3.3.1 for Blackwell kernel compilation.
# If this fails on Windows, it's OK — we'll use attn_implementation="sdpa" instead.
!pip install "triton>=3.3.1"

ERROR: Could not find a version that satisfies the requirement triton>=3.3.1 (from versions: none)

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for triton>=3.3.1


## 4. Install bitsandbytes

In [4]:
!pip install bitsandbytes

try:
    import bitsandbytes as bnb
    print(f"bitsandbytes: {bnb.__version__}")
except Exception as e:
    print(f"⚠ bitsandbytes import failed: {e}")
    print("  4-bit QLoRA may not work. Fall back to load_in_8bit or try building from source.")


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/55.4 MB ? eta -:--:--
   - -------------------------------------- 2.4/55.4 MB 19.1 MB/s eta 0:00:03
   ------------- -------------------------- 18.6/55.4 MB 58.6 MB/s eta 0:00:01
   ------------------------- -------------- 35.9/55.4 MB 69.1 MB/s eta 0:00:01
   ------------------------------------- -- 51.4/55.4 MB 69.5 MB/s eta 0:00:01
   ---------------------------------------- 55.4/55.4 MB 59.8 MB/s eta 0:00:00


W0323 17:20:17.196000 34628 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


bitsandbytes: 0.49.2


## 5. Install unsloth

In [5]:
!pip install unsloth

try:
    from unsloth import FastLanguageModel
    print("✓ unsloth imported successfully")
except Exception as e:
    print(f"⚠ unsloth import failed: {e}")
    print("  Check https://docs.unsloth.ai/basics/installation for troubleshooting.")


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
[unsloth.import_fixes|WARNING]Unsloth: torch==2.12.0.dev20260323+cu128 requires torchvision>=0.27.0, but found torchvision==0.26.0.dev20260323+cu128. Try updating torchvision via `pip install --upgrade "torchvision>=0.27.0"`. Please refer to https://pytorch.org/get-started/previous-versions/ for more information.
Detected a pre-release build. Continuing with a warning. Set UNSLOTH_SKIP_TORCHVISION_CHECK=1 to silence this.


  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached fsspec-2025.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/62.3 MB ? eta -:--:--
   -- ------------------------------------- 4.2/62.3 MB 27.9 MB/s eta 0:00:03
   ------------- -------------------------- 20.7/62.3 MB 65.5 MB/s eta 0:00:01
   ------------------------- -------------- 39.3/62.3 MB 71.4 MB/s eta 0:00:01
   ---------------------------------- ----- 53.5/62.3 MB 71.0 MB/s eta 0:00:01
   ---------------------------------------  62.1/62.3 MB 66.0 MB/s eta 0:00:01
   ---------------------------------------- 62.3/62.3 MB 58.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/557.0 kB ? eta -:--:--
   --------------------------------------- 557.0/557.0 kB 19.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   --------------------------

c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0323 17:22:24.495000 34628 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchao\float8\float8_training_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchao\float8\float8_training_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  

Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
✓ unsloth imported successfully


## 6. Install remaining dependencies

In [6]:
!pip install trl datasets anthropic pydantic


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 7. Full Dependency Verification

In [7]:
import importlib

deps = {
    "torch": "torch",
    "unsloth": "unsloth",
    "trl": "trl",
    "transformers": "transformers",
    "datasets": "datasets",
    "bitsandbytes": "bitsandbytes",
    "peft": "peft",
    "accelerate": "accelerate",
    "anthropic": "anthropic",
    "pydantic": "pydantic",
    "triton": "triton",
}

all_ok = True
print(f"{'Package':<20} {'Version':<20} {'Status'}")
print("-" * 50)
for name, module in deps.items():
    try:
        m = importlib.import_module(module)
        v = getattr(m, "__version__", "?")
        print(f"{name:<20} {v:<20} ✓")
    except ImportError:
        print(f"{name:<20} {'—':<20} ✗ MISSING")
        all_ok = False

print()
if all_ok:
    print("✓ All dependencies installed. Ready for training.")
else:
    print("✗ Some dependencies missing. Review errors above.")

Package              Version              Status
--------------------------------------------------
torch                2.12.0.dev20260323+cu128 ✓
unsloth              2026.3.10            ✓
trl                  0.24.0               ✓
transformers         5.3.0                ✓
datasets             4.3.0                ✓
bitsandbytes         0.49.2               ✓
peft                 0.18.1               ✓
accelerate           1.13.0               ✓
anthropic            0.86.0               ✓
pydantic             2.12.5               ✓
triton               3.6.0                ✓

✓ All dependencies installed. Ready for training.


## 8. Quick unsloth + Qwen3-8B Load Test

Verifica que unsloth puede cargar el modelo base. Esto descarga ~5GB la primera vez.

In [9]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3.5-9B",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,  # auto-detect
)

free, total = torch.cuda.mem_get_info(0)
used = (total - free) / 1e9
print(f"\nModel loaded. VRAM used: {used:.1f} GB / {total/1e9:.1f} GB")
print(f"VRAM free: {free/1e9:.1f} GB")
print("\n✓ Qwen3-8B loads correctly with unsloth. Ready for fine-tuning.")

# Cleanup
del model, tokenizer
torch.cuda.empty_cache()

==((====))==  Unsloth 2026.3.10: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 5070 Ti. Num GPUs = 1. Max memory: 15.92 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.12.0.dev20260323+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights:   0%|          | 2/760 [00:02<12:30,  1.01it/s]c:\Users\sandy\AppData\Local\Programs\Python\Python312\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 760/760 [00:13<00:00, 55.50it/s] 



Model loaded. VRAM used: 12.1 GB / 17.1 GB
VRAM free: 5.0 GB

✓ Qwen3-8B loads correctly with unsloth. Ready for fine-tuning.
